In [1]:
import sys
import json
from pathlib import Path
from datetime import datetime

from IPython.display import display, Markdown

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from config.settings import (
    PDF_STRUCTURE_PATH,
    CLEAN_TEXT_PATH,
    TEXT_CHUNKS_PATH,
    TEXT_DIR,
    CHUNK_TARGET_SIZE,
    CHUNK_MAX_SIZE,
    CHUNK_OVERLAP,
    SKIP_SECTIONS,
)

TEXT_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
assert PDF_STRUCTURE_PATH.exists(), f"Not found: {PDF_STRUCTURE_PATH}"
assert CLEAN_TEXT_PATH.exists(), f"Not found: {CLEAN_TEXT_PATH}"

structure = json.loads(PDF_STRUCTURE_PATH.read_text(encoding="utf-8"))
clean_pages = json.loads(CLEAN_TEXT_PATH.read_text(encoding="utf-8"))

toc = structure["toc"]
total_pdf_pages = structure["total_pages"]

# Page lookup: page number -> clean_text
page_lookup = {p["page"]: p["clean_text"] for p in clean_pages}

display(Markdown(f"""
### Inputs Loaded

| Item | Value |
|------|-------|
| **TOC entries** | {len(toc)} |
| **Clean pages** | {len(clean_pages)} |
| **Total PDF pages** | {total_pdf_pages} |
""".strip()))

### Inputs Loaded

| Item | Value |
|------|-------|
| **TOC entries** | 1224 |
| **Clean pages** | 1314 |
| **Total PDF pages** | 1314 |

In [3]:
def build_sections(toc, total_pages):
    sections = []
    path_stack = {}

    for i, entry in enumerate(toc):
        level = entry["level"]
        title = entry["title"].strip()
        start_page = entry["page"]

        # Update path stack at current level, clear deeper levels
        path_stack[level] = title
        path_stack = {k: v for k, v in path_stack.items() if k <= level}

        # Section path from level 1 to current level
        section_path = [path_stack[l] for l in sorted(path_stack.keys())]

        # End page: next TOC entry start page - 1, or last PDF page
        if i + 1 < len(toc):
            end_page = toc[i + 1]["page"] - 1
            # If next entry starts on the same page, end = start
            if end_page < start_page:
                end_page = start_page
        else:
            end_page = total_pages

        sections.append({
            "toc_index": i,
            "toc_level": level,
            "section_title": title,
            "section_path": section_path,
            "section_start_page": start_page,
            "section_end_page": end_page,
        })

    return sections


sections = build_sections(toc, total_pdf_pages)

display(Markdown(f"### TOC Sections Built: {len(sections)}"))

### TOC Sections Built: 1224

In [4]:
def collect_section_text(section, page_lookup):
    start = section["section_start_page"]
    end = section["section_end_page"]
    pages_text = []
    source_pages = []

    for pg in range(start, end + 1):
        text = page_lookup.get(pg, "").strip()
        if text:
            pages_text.append(text)
            source_pages.append(pg)

    return "\n\n".join(pages_text), source_pages


section_texts = []

for sec in sections:
    # Skip configured sections (e.g., "Contents")
    if sec["section_title"] in SKIP_SECTIONS:
        continue

    text, source_pages = collect_section_text(sec, page_lookup)
    if not text.strip():
        continue

    section_texts.append({
        **sec,
        "text": text,
        "source_pages": source_pages,
    })

display(Markdown(f"""
### Section Text Collected

| Item | Value |
|------|-------|
| **Total TOC sections** | {len(sections)} |
| **Sections with text** | {len(section_texts)} |
| **Skipped sections** | {len(sections) - len(section_texts)} |
""".strip()))

### Section Text Collected

| Item | Value |
|------|-------|
| **Total TOC sections** | 1224 |
| **Sections with text** | 1223 |
| **Skipped sections** | 1 |

In [5]:
def split_section_text(text, target=CHUNK_TARGET_SIZE, maximum=CHUNK_MAX_SIZE, overlap=CHUNK_OVERLAP):
    if len(text) <= maximum:
        return [text]

    paragraphs = text.split("\n\n")
    chunks = []
    current = ""

    for para in paragraphs:
        # If adding this paragraph stays within target, accumulate
        if len(current) + len(para) + 2 <= maximum:
            current = f"{current}\n\n{para}".strip()
        else:
            if current:
                chunks.append(current)
                # Overlap: take last N characters from current chunk
                tail = current[-overlap:] if overlap else ""
                current = f"{tail}\n\n{para}".strip() if tail else para
            else:
                # Single paragraph exceeds max — force-split by lines
                lines = para.split("\n")
                for line in lines:
                    if len(current) + len(line) + 1 <= maximum:
                        current = f"{current}\n{line}".strip()
                    else:
                        if current:
                            chunks.append(current)
                        current = line

    if current.strip():
        chunks.append(current.strip())

    return chunks


split_sections = []

for sec in section_texts:
    parts = split_section_text(sec["text"])
    for idx, part_text in enumerate(parts):
        split_sections.append({
            **{k: v for k, v in sec.items() if k != "text"},
            "text": part_text,
            "part_index": idx + 1,
            "part_count": len(parts),
        })

display(Markdown(f"""
### Chunking Complete

| Item | Value |
|------|-------|
| **Sections input** | {len(section_texts)} |
| **Chunks output** | {len(split_sections)} |
| **Multi-part sections** | {sum(1 for s in section_texts if len(split_section_text(s['text'])) > 1)} |
""".strip()))

### Chunking Complete

| Item | Value |
|------|-------|
| **Sections input** | 1223 |
| **Chunks output** | 1391 |
| **Multi-part sections** | 99 |

In [6]:
chunks = []

for i, sec in enumerate(split_sections):
    section_prefix = " > ".join(sec["section_path"])
    full_text = f"Section: {section_prefix}\n\n{sec['text']}"
    source_pages = sec["source_pages"]

    chunks.append({
        "chunk_id": f"chunk_{i:06d}",
        "chunk_index": i,
        "text": full_text,
        "char_count": len(full_text),
        "word_count": len(full_text.split()),
        "page_start": source_pages[0] if source_pages else sec["section_start_page"],
        "page_end": source_pages[-1] if source_pages else sec["section_end_page"],
        "source_pages": source_pages,
        "toc_level": sec["toc_level"],
        "section_title": sec["section_title"],
        "section_path": sec["section_path"],
        "section_start_page": sec["section_start_page"],
        "section_end_page": sec["section_end_page"],
        "part_index": sec["part_index"],
        "part_count": sec["part_count"],
    })

display(Markdown(f"### Final Chunk Records: {len(chunks)}"))

### Final Chunk Records: 1391

In [7]:
TEXT_CHUNKS_PATH.write_text(
    json.dumps(chunks, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

display(Markdown(f"""
### Chunks Saved

| Item | Value |
|------|-------|
| **Total chunks** | {len(chunks)} |
| **Output** | `{TEXT_CHUNKS_PATH.relative_to(project_root)}` |
""".strip()))

### Chunks Saved

| Item | Value |
|------|-------|
| **Total chunks** | 1391 |
| **Output** | `SCADA-DIP\data\text\text_chunks.json` |

In [ ]:
char_counts = [c["char_count"] for c in chunks]
avg_chars = sum(char_counts) / len(char_counts) if char_counts else 0
min_chars = min(char_counts) if char_counts else 0
max_chars = max(char_counts) if char_counts else 0
below_500 = sum(1 for c in char_counts if c < 500)
above_3500 = sum(1 for c in char_counts if c > 3500)

all_pages = set()
for c in chunks:
    all_pages.update(c["source_pages"])
page_coverage = len(all_pages)

# Sample chunks
sample_indices = [0, len(chunks) // 2]
sample_blocks = ""
for idx in sample_indices:
    c = chunks[idx]
    preview = c["text"][:400] + ("..." if len(c["text"]) > 400 else "")
    sample_blocks += f"""
---
**{c['chunk_id']}** | Level {c['toc_level']} | Pages {c['page_start']}-{c['page_end']} | {c['char_count']} chars | Part {c['part_index']}/{c['part_count']}

Path: `{' > '.join(c['section_path'])}`

```
{preview}
```
"""

display(Markdown(f"""
### Chunking Quality Summary

| Item | Value |
|------|-------|
| **Total TOC sections** | {len(section_texts)} |
| **Total chunks** | {len(chunks)} |
| **Avg chunk size** | {avg_chars:.0f} chars |
| **Min chunk size** | {min_chars} chars |
| **Max chunk size** | {max_chars} chars |
| **Chunks < 500 chars** | {below_500} |
| **Chunks > 3500 chars** | {above_3500} |
| **Pages covered** | {page_coverage} / {total_pdf_pages} |
| **Output** | `{TEXT_CHUNKS_PATH.relative_to(project_root)}` |
| **Timestamp** | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} |

### Sample Chunks
{sample_blocks}
""".strip()))

### Chunking Quality Summary

| Item | Value |
|------|-------|
| **Total TOC sections** | 1223 |
| **Total chunks** | 1391 |
| **Avg chunk size** | 2008 chars |
| **Min chunk size** | 435 chars |
| **Max chunk size** | 3622 chars |
| **Chunks < 500 chars** | 3 |
| **Chunks > 3500 chars** | 16 |
| **Pages covered** | 1281 / 1314 |
| **Output** | `SCADA-DIP\data\text\text_chunks.json` |
| **Timestamp** | 2026-04-24 09:05:50 |

### Sample Chunks

---
**chunk_000000** | Level 1 | Pages 33-34 | 1773 chars | Part 1/1

Path: `Safety Precautions`

```
Section: Safety Precautions

Safety Precautions
During installation or use of this software, pay attention to all safety messages that occur in the
software and that are included in the documentation. The following safety messages apply to this
software in its entirety.
WARNING
UNINTENDED EQUIPMENT OPERATION
• Do not use the software or devices for critical control or protection applications where...
```

---
**chunk_000695** | Level 3 | Pages 705-705 | 1158 chars | Part 1/1

Path: `Operate > Use Security Viewer > Security Viewer Filter`

```
Section: Operate > Use Security Viewer > Security Viewer Filter

To add a column:
Right-click in the header area of the log, then choose Insert Column. From the list that
appears, check an additional column title. The new column displays to the left of the column
you clicked.
To remove a column:
Right-click on the header of the column you want to delete and then click Remove Column.
3. You can fil...
```

In [10]:
# Uncovered pages
all_covered = set()
for c in chunks:
    all_covered.update(c["source_pages"])
uncovered = sorted(set(range(1, total_pdf_pages + 1)) - all_covered)

uncovered_display = ", ".join(str(p) for p in uncovered[:50])
if len(uncovered) > 50:
    uncovered_display += f" ... ({len(uncovered)} total)"

# Small chunks
small_chunks = [c for c in chunks if c["char_count"] < 500]
small_rows = "\n".join(
    f"| `{c['chunk_id']}` | {c['section_title'][:50]} | {c['page_start']}-{c['page_end']} | {c['char_count']} |"
    for c in small_chunks[:15]
)

# Large chunks
large_chunks = [c for c in chunks if c["char_count"] > 3500]
large_rows = "\n".join(
    f"| `{c['chunk_id']}` | {c['section_title'][:50]} | {c['page_start']}-{c['page_end']} | {c['char_count']} |"
    for c in large_chunks[:15]
)

# Top 10 largest
top10 = sorted(chunks, key=lambda c: c["char_count"], reverse=True)[:10]
top10_rows = "\n".join(
    f"| `{c['chunk_id']}` | {c['section_title'][:50]} | {c['page_start']}-{c['page_end']} | {c['char_count']} |"
    for c in top10
)

display(Markdown(f"""
### Chunk Validation

#### Uncovered Pages ({len(uncovered)} pages)

{uncovered_display if uncovered else "All pages covered."}

---

#### Chunks Below 500 Characters ({len(small_chunks)} chunks, showing up to 15)

| Chunk ID | Section Title | Pages | Chars |
|----------|--------------|-------|-------|
{small_rows if small_rows else "| — | None | — | — |"}

---

#### Chunks Above 3500 Characters ({len(large_chunks)} chunks, showing up to 15)

| Chunk ID | Section Title | Pages | Chars |
|----------|--------------|-------|-------|
{large_rows if large_rows else "| — | None | — | — |"}

---

#### Top 10 Largest Chunks

| Chunk ID | Section Title | Pages | Chars |
|----------|--------------|-------|-------|
{top10_rows}
""".strip()))

### Chunk Validation

#### Uncovered Pages (33 pages)

1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 35

---

#### Chunks Below 500 Characters (3 chunks, showing up to 15)

| Chunk ID | Section Title | Pages | Chars |
|----------|--------------|-------|-------|
| `chunk_000091` | Time synchronization | 114-115 | 480 |
| `chunk_000232` | Summary | 213-213 | 435 |
| `chunk_001276` | Graphics Editor File Menu — Print Submenu | 1189-1189 | 456 |

---

#### Chunks Above 3500 Characters (16 chunks, showing up to 15)

| Chunk ID | Section Title | Pages | Chars |
|----------|--------------|-------|-------|
| `chunk_000393` | Email basic reports | 321-323 | 3562 |
| `chunk_000502` | Control snippet example | 506-518 | 3567 |
| `chunk_000534` | Adding a new trend | 570-571 | 3577 |
| `chunk_000550` | Web Applications Access and Privilege Levels | 583-584 | 3619 |
| `chunk_000601` | Add Advanced Dashboards and Reports into Web Appli | 628-631 | 3561 |
| `chunk_000669` | Web Applications Access and Privilege Levels | 687-688 | 3622 |
| `chunk_000713` | Create and view basic reports | 719-721 | 3532 |
| `chunk_000835` | Upgrade Information for versions 8.1 and 8.0 SR1 | 835-838 | 3554 |
| `chunk_000865` | Citect.ini parameters in 7.30 | 869-873 | 3571 |
| `chunk_000960` | Logic code definitions | 907-939 | 3527 |
| `chunk_000963` | Logic code definitions | 907-939 | 3524 |
| `chunk_001083` | TGML DOM Methods | 1015-1016 | 3521 |
| `chunk_001148` | TGML Element Summary | 1086-1097 | 3579 |
| `chunk_001194` | Attributes Overview | 1129-1130 | 3573 |
| `chunk_001340` | Repair one-line diagrams | 1251-1252 | 3612 |

---

#### Top 10 Largest Chunks

| Chunk ID | Section Title | Pages | Chars |
|----------|--------------|-------|-------|
| `chunk_000669` | Web Applications Access and Privilege Levels | 687-688 | 3622 |
| `chunk_000550` | Web Applications Access and Privilege Levels | 583-584 | 3619 |
| `chunk_001340` | Repair one-line diagrams | 1251-1252 | 3612 |
| `chunk_001148` | TGML Element Summary | 1086-1097 | 3579 |
| `chunk_000534` | Adding a new trend | 570-571 | 3577 |
| `chunk_001194` | Attributes Overview | 1129-1130 | 3573 |
| `chunk_000865` | Citect.ini parameters in 7.30 | 869-873 | 3571 |
| `chunk_000502` | Control snippet example | 506-518 | 3567 |
| `chunk_000393` | Email basic reports | 321-323 | 3562 |
| `chunk_000601` | Add Advanced Dashboards and Reports into Web Appli | 628-631 | 3561 |